# 04 · Artigos bons, destacados e amostragem

O benchmark usado na pesquisa tem **100 artigos bons e 100 destacados**. A seleção original utilizou uma semente fixa (`random_state=42`), para permitir refazer o sorteio sobre a mesma população, na mesma ordem. A lista completa dos candidatos daquele dia não está nos arquivos finais; por isso, o modo padrão preserva a amostra histórica, sem produzir silenciosamente um sorteio diferente.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRUTOS = ROOT / 'dados' / 'brutos'
PROCESSADOS = ROOT / 'dados' / 'processados'
GRAFICOS = ROOT / 'resultados' / 'graficos'
PROCESSADOS.mkdir(parents=True, exist_ok=True)
GRAFICOS.mkdir(parents=True, exist_ok=True)

import requests, time, re
from datetime import datetime

FONTE = BRUTOS/'resultado_tcc_benchmark_artigos_top.xlsx'
benchmark = pd.read_excel(FONTE,sheet_name='Benchmark_200')
sobreposicoes = pd.read_excel(FONTE,sheet_name='Sobreposicoes')
resumo_original = pd.read_excel(FONTE,sheet_name='Resumo_Grupos')
assert len(benchmark)==200
assert benchmark['tipo'].value_counts().to_dict()=={'Bom':100,'Destacado':100}
assert benchmark['pageid'].is_unique
print('Benchmark histórico carregado:',benchmark['tipo'].value_counts().to_dict())

## Sorteio reprodutível com seed 42

Uma semente faz o gerador pseudoaleatório iniciar sempre no mesmo estado. **Não** significa que qualquer listagem futura produzirá os mesmos 100 artigos: a população elegível e sua ordem também precisam ser iguais. Os artigos presentes nas duas classificações ficam no universo de destacados; são removidos do universo de bons antes da seleção.

In [ ]:
SEED = 42
TAMANHO_POR_TIPO = 100
POPULACOES = BRUTOS/'populacoes_benchmark_2026.xlsx'  # opcional: lista completa congelada, se disponível


def sortear_artigos(bons, destacados):
    """Devolve duas amostras de 100, com sobreposições removidas dos bons."""
    b=bons[['pageid','title']].drop_duplicates('pageid').copy()
    d=destacados[['pageid','title']].drop_duplicates('pageid').copy()
    b=b[~b['pageid'].isin(d['pageid'])].copy()
    # A ordem abaixo deve coincidir com a ordem histórica para reproduzir a amostra original.
    b=b.sort_values(['pageid']).reset_index(drop=True)
    d=d.sort_values(['pageid']).reset_index(drop=True)
    assert len(b)>=TAMANHO_POR_TIPO and len(d)>=TAMANHO_POR_TIPO
    amostra_b=b.sample(n=TAMANHO_POR_TIPO,random_state=SEED).assign(tipo='Bom')
    amostra_d=d.sample(n=TAMANHO_POR_TIPO,random_state=SEED).assign(tipo='Destacado')
    return pd.concat([amostra_b,amostra_d],ignore_index=True)

if POPULACOES.exists():
    bens = pd.read_excel(POPULACOES,sheet_name='Bons')
    destaques = pd.read_excel(POPULACOES,sheet_name='Destacados')
    sorteio_exemplo=sortear_artigos(bens,destaques)
    print('Sorteio da população informada:',len(sorteio_exemplo))
else:
    print('Sem listas históricas completas: mantida a amostra original de 200 artigos.')
    print('A semente 42 está documentada; não foi gerado novo sorteio.')

## Consulta de artigos selecionados pela API

Com `ATUALIZAR_API = True`, o código recupera as revisões correntes dos mesmos `pageid`, sem trocar os artigos selecionados. Essa atualização produz um arquivo separado e não substitui o benchmark de 2026.

In [ ]:
ATUALIZAR_API = False
API='https://pt.wikipedia.org/w/api.php'
AGENTE='TCC-Wikipedia-Qualidade/1.0 (contato: seu-email@exemplo.com)'


def obter_artigo(pageid, session):
    parametros={'action':'query','format':'json','formatversion':2,'prop':'revisions|pageprops',
        'pageids':int(pageid),'rvprop':'ids|timestamp|size|content','rvslots':'main','rvlimit':1,'ppprop':'wikibase_item'}
    for tentativa in range(6):
        res=session.get(API,params=parametros,timeout=45)
        if res.status_code in (429,500,502,503,504):
            time.sleep(2**tentativa);continue
        res.raise_for_status()
        pagina=res.json().get('query',{}).get('pages',[{}])[0]
        rev=pagina.get('revisions',[{}])[0]
        return {'pageid':pageid,'title':pagina.get('title'),
                'revid':rev.get('revid'),'timestamp':rev.get('timestamp'),
                'bytes_total':rev.get('size'),
                'qid_wikidata':pagina.get('pageprops',{}).get('wikibase_item'),
                'wikitext':rev.get('slots',{}).get('main',{}).get('content','')}
    raise RuntimeError(f'Falha na consulta do artigo {pageid}')

if ATUALIZAR_API:
    assert 'seu-email@exemplo.com' not in AGENTE, 'Informe um contato no User-Agent.'
    sessao=requests.Session();sessao.headers['User-Agent']=AGENTE
    itens=[obter_artigo(x,sessao) for x in benchmark['pageid']]
    atual = pd.DataFrame(itens)
    atual.to_excel(PROCESSADOS/'benchmark_recoleta_atual.xlsx',index=False)

## Extração das métricas na recoleta (opcional)

Para atualizar os artigos selecionados, as métricas são calculadas sobre o wikitext retornado pela API. As definições de referências e links são regras quantitativas, não avaliações humanas do conteúdo. A atualização é salva separadamente da amostra histórica.

In [ ]:
REF_ABERTURA = re.compile(r'<ref\b[^>]*>',re.I)
REF_TEXTO = re.compile(r'<ref\b[^>]*>.*?</ref\s*>|<ref\b[^>]*/>',re.I|re.S)
SECAO = re.compile(r'^={2,6}\s*[^=\n]+?\s*={2,6}\s*$',re.M)
LINK_INTERNO = re.compile(r'\[\[([^]\n]+)\]\]')

def extrair_metricas(txt, tamanho):
    sem_ref=REF_TEXTO.sub('',txt)
    links=LINK_INTERNO.findall(txt)
    links_artigos=[x for x in links if not re.match(r'\s*(?:Categoria|Arquivo|Ficheiro|File|Image)\s*:',x,re.I)]
    return {
        'bytes_total':int(tamanho),
        'referencias':len(REF_ABERTURA.findall(txt)),
        'secoes':len(SECAO.findall(txt)),
        'infobox':int(bool(re.search(r'\{\{\s*(?:Infobox|Info/)',txt,re.I))),
        'ver_tambem':int(bool(re.search(r'^={2,6}\s*Ver também\s*={2,6}',txt,re.I|re.M))),
        'links_internos':len(links_artigos),
        'links_externos':len(re.findall(r'https?://[^\s\]<>|{}]+',sem_ref,re.I)),
        'categorias':len(re.findall(r'\[\[\s*Categoria\s*:',txt,re.I)),
        'imagens':len(re.findall(r'\[\[\s*(?:Arquivo|Ficheiro|File|Image)\s*:',txt,re.I)),
    }

if ATUALIZAR_API:
    # A etapa anterior criou a lista `itens` com wikitext e metadados.
    metricas_atualizadas=[]
    for item in itens:
        medido=extrair_metricas(item['wikitext'],item['bytes_total'] or 0)
        metricas_atualizadas.append({k:v for k,v in item.items() if k!='wikitext'} | medido)
    pd.DataFrame(metricas_atualizadas).to_excel(
        PROCESSADOS/'benchmark_metricas_recoleta_atual.xlsx',index=False)
    print('Métricas atuais medidas:',len(metricas_atualizadas))

## Parâmetros e arquivo final do benchmark

Os valores usados nos indicadores vêm do conjunto de artigos bons. Destacados compõem um segundo conjunto de comparação. A planilha congelada mantém os 200 identificadores e as métricas observadas durante a pesquisa.

In [ ]:
metricas=['bytes_total','referencias','secoes','links_internos','links_externos','categorias','imagens']
medianas=benchmark.groupby('tipo')[metricas].median().round(2)
display(medianas)
assert abs(float(medianas.loc['Bom','bytes_total'])-25843)<1e-9
assert len(sobreposicoes)==4

saida=PROCESSADOS/'resultado_tcc_benchmark_artigos_top.xlsx'
with pd.ExcelWriter(saida,engine='openpyxl') as writer:
    benchmark.to_excel(writer,sheet_name='Benchmark_200',index=False)
    resumo_original.to_excel(writer,sheet_name='Resumo_Grupos',index=False)
    sobreposicoes.to_excel(writer,sheet_name='Sobreposicoes',index=False)
print('Arquivo gerado:',saida.name)